## Crawling Data Detik.com

Pada bagian ini, kita akan berlatih melakukan proses pengambilan data (*web crawling*) dengan mengumpulkan judul-judul berita olahraga terkini dari situs **Detik Health**.

Untuk melakukan ekstraksi data ini, terdapat tiga *library* utama yang akan digunakan:
* **Requests:** Berfungsi untuk mengirimkan permintaan (HTTP Request) ke server website dan mengambil kode HTML mentah darinya.
* **BeautifulSoup:** Berfungsi untuk mengurai dan menyusun struktur HTML sehingga elemen-elemen tertentu (seperti teks judul dan link) dapat dipilah dengan mudah.
* **Pandas:** Berfungsi untuk mengorganisir hasil data yang telah diekstrak ke dalam bentuk tabel (*DataFrame*) yang terstruktur rapi.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Menentukan URL target (Detik Sport) dan Headers
url = 'https://health.detik.com/indeks'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
}

# 2. Mengambil konten HTML dari website
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# 3. Mencari semua blok artikel berita
articles = soup.find_all('article')
data_berita = []

# 4. Mengekstrak judul dan tautan dari masing-masing artikel
for article in articles:
    title_tag = article.find('h3')
    link_tag = article.find('a')
    
    if title_tag and link_tag:
        judul = title_tag.get_text(strip=True)
        tautan = link_tag['href']
        
        data_berita.append({
            'Judul Berita': judul,
            'Tautan': tautan
        })

# 5. Menampilkan hasil dalam bentuk tabel Pandas
df_berita = pd.DataFrame(data_berita)
df_berita.head(10) # Menampilkan 10 berita teratas

""


Data tabular di atas merupakan hasil akhir dari proses pengumpulan informasi mentah pada halaman indeks. Setelah dikumpulkan dalam format tabel seperti ini, dataset sudah siap untuk dibersihkan dan diproses lebih lanjut pada tahap prapemrosesan teks (*text preprocessing*) dalam siklus Web Mining.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

headers = {
    'User-Agent': 'Mozilla/5.0'
}

url = 'https://health.detik.com/indeks'

data_berita = []

# Maksimal 10 halaman
for halaman in range(10):

    print(f"Mengambil halaman {halaman + 1}...")

    response = requests.get(
        url,
        headers=headers,
        timeout=10
    )

    if response.status_code != 200:
        print("Gagal mengakses halaman")
        break

    soup = BeautifulSoup(response.text, 'html.parser')

    articles = soup.find_all('article')

    print("Artikel ditemukan:", len(articles))

    for article in articles:

        title_tag = article.find('h3')
        link_tag = article.find('a')

        if title_tag and link_tag:

            judul = title_tag.get_text(strip=True)
            tautan = urljoin(
                url,
                link_tag.get('href')
            )

            data_berita.append({
                'Judul Berita': judul,
                'Tautan': tautan
            })

        if len(data_berita) >= 200:
            break

    if len(data_berita) >= 200:
        break

    # Cari link Next
    next_link = soup.find(
        'a',
        string=lambda x: x and x.strip().lower() == 'next'
    )

    if next_link:
        url = urljoin(url, next_link.get('href'))
    else:
        print("Halaman berikutnya tidak ditemukan.")
        break


# Membuat DataFrame
df_berita = pd.DataFrame(data_berita)

# Hapus duplikat
df_berita = df_berita.drop_duplicates()

# Batasi 200 data
df_berita = df_berita.head(200)

# Nomor urut
df_berita.index = range(1, len(df_berita) + 1)
df_berita.index.name = 'No'

print("\nJumlah berita:", len(df_berita))

display(df_berita)

Mengambil halaman 1...
Gagal mengakses halaman

Jumlah berita: 0


""
No


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

headers = {
    'User-Agent': 'Mozilla/5.0'
}

url = 'https://health.detik.com/indeks'


data_berita = []


for halaman in range(10):

    print(f"Mengambil halaman {halaman + 1}...")

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        response.raise_for_status()

    except requests.exceptions.RequestException as e:
        print("Terjadi kesalahan:", e)
        break

    # Parsing HTML
    soup = BeautifulSoup(response.text, 'html.parser')

    # Mencari semua artikel
    articles = soup.find_all('article')

    print("Artikel ditemukan:", len(articles))

    # Mengambil judul dan tautan
    for article in articles:

        title_tag = article.find('h3')
        link_tag = article.find('a')

        if title_tag and link_tag:

            judul = title_tag.get_text(strip=True)
            tautan = link_tag.get('href')

            # Jika tautan kosong, lanjutkan
            if not tautan:
                continue

            # Mengubah tautan menjadi URL lengkap
            tautan = urljoin(url, tautan)

            # Menambahkan data berita
            data_berita.append({
                'Judul Berita': judul,
                'Tautan': tautan
            })

        # Berhenti jika sudah mendapatkan 200 berita
        if len(data_berita) >= 200:
            break

    # Jika sudah mendapatkan 200 data
    if len(data_berita) >= 200:
        break

    # =========================
    # MENCARI LINK HALAMAN BERIKUTNYA
    # =========================

    next_link = None

    for link in soup.find_all('a'):

        teks_link = link.get_text(strip=True).lower()

        if teks_link == 'next':
            next_link = link.get('href')
            break

    # Jika tidak ada halaman berikutnya
    if not next_link:
        print("Halaman berikutnya tidak ditemukan.")
        break

    # Pindah ke halaman berikutnya
    url = urljoin(url, next_link)


if len(data_berita) == 0:
    print("\nTidak berhasil mengambil data dari health.detik.com")
    print("(kemungkinan akses ke situs tersebut diblokir oleh jaringan pada environment ini).")
    print("Menggunakan DATA CONTOH (dummy) sebagai pengganti, agar seluruh")
    print("tahapan selanjutnya tetap bisa dijalankan & didemonstrasikan.\n")

    from sample_data import SAMPLE_BERITA
    data_berita = [
        {'Judul Berita': judul, 'Tautan': tautan}
        for judul, tautan in SAMPLE_BERITA
    ]

df_berita = pd.DataFrame(data_berita)

# Menghapus data duplikat
df_berita = df_berita.drop_duplicates()

# Membatasi maksimal 200 data
df_berita = df_berita.head(200)

# Membuat nomor urut
df_berita.index = range(1, len(df_berita) + 1)
df_berita.index.name = 'No'


print("\nJumlah data berita:", len(df_berita))

print("\n========== 10 DATA BERITA PERTAMA ==========")
display(df_berita.head(10))

print("\n========== 10 DATA BERITA TERAKHIR ==========")
display(df_berita.tail(10))


# Menggabungkan semua judul berita
semua_judul = ' '.join(df_berita['Judul Berita'])

# Mengubah menjadi huruf kecil dan mengambil kata
kata = re.findall(r'\b\w+\b', semua_judul.lower())

# Menghilangkan angka
kata = [k for k in kata if not k.isdigit()]

# Mengambil semua kata unik
kata_unik = list(set(kata))

# Mengurutkan kata unik agar rapi
kata_unik.sort()

df_kata_unik = pd.DataFrame(
    kata_unik,
    columns=['Kata Unik']
)

# Membuat nomor urut
df_kata_unik.index = range(1, len(df_kata_unik) + 1)
df_kata_unik.index.name = 'No'

print("\nJumlah seluruh kata unik:", len(df_kata_unik))

print("\n========== 10 KATA UNIK PERTAMA ==========")
display(df_kata_unik.head(10))

print("\n========== 10 KATA UNIK TERAKHIR ==========")
display(df_kata_unik.tail(10))

# =========================================================
# Menyimpan hasil crawling ke CSV agar bisa dipakai kembali
# oleh notebook Tahap 2 (Text Preprocessing, TF-IDF, PCA)
# tanpa harus meng-crawl ulang.
# =========================================================
df_berita.to_csv('df_berita_hasil_crawling.csv', index=True)
print("\nData berhasil disimpan ke 'df_berita_hasil_crawling.csv'")


Mengambil halaman 1...
Terjadi kesalahan: 403 Client Error: Forbidden for url: https://health.detik.com/indeks

Tidak berhasil mengambil data dari health.detik.com
(kemungkinan akses ke situs tersebut diblokir oleh jaringan pada environment ini).
Menggunakan DATA CONTOH (dummy) sebagai pengganti, agar seluruh
tahapan selanjutnya tetap bisa dijalankan & didemonstrasikan.




Jumlah data berita: 40

========== 10 DATA BERITA PERTAMA ==========


,Judul Berita,Tautan
No,,
1,Studi Terbaru Ungkap Manfaat Jalan Kaki 30 Menit Setiap Hari,https://health.detik.com/berita-detikhealth/d-0001
2,Kemenkes Ingatkan Warga Waspada Lonjakan Kasus DBD di Musim Hujan,https://health.detik.com/berita-detikhealth/d-0002
3,5 Kebiasaan Sederhana yang Bisa Menurunkan Risiko Diabetes,https://health.detik.com/berita-detikhealth/d-0003
4,Dokter Jelaskan Perbedaan Flu Biasa dan Flu Singapura pada Anak,https://health.detik.com/berita-detikhealth/d-0004
5,Riset: Kurang Tidur Bisa Tingkatkan Risiko Penyakit Jantung,https://health.detik.com/berita-detikhealth/d-0005
6,BPOM Rilis Daftar Obat Sirup yang Aman Dikonsumsi Anak,https://health.detik.com/berita-detikhealth/d-0006
7,Tips Menjaga Kesehatan Mental di Tengah Padatnya Pekerjaan,https://health.detik.com/berita-detikhealth/d-0007
8,"Kasus Cacar Monyet Bertambah, Ini Gejala yang Perlu Diwaspadai",https://health.detik.com/berita-detikhealth/d-0008
9,Ahli Gizi Sarankan Konsumsi Serat untuk Jaga Kesehatan Usus,https://health.detik.com/berita-detikhealth/d-0009



========== 10 DATA BERITA TERAKHIR ==========


,Judul Berita,Tautan
No,,
31,Cara Mencegah Penularan TBC di Lingkungan Rumah,https://health.detik.com/berita-detikhealth/d-0031
32,Kenali Tanda-tanda Kelelahan Kerja atau Burnout,https://health.detik.com/berita-detikhealth/d-0032
33,Manfaat Yoga untuk Fleksibilitas dan Kesehatan Tulang,https://health.detik.com/berita-detikhealth/d-0033
34,Peneliti Temukan Kaitan Kurang Gerak dengan Risiko Diabetes,https://health.detik.com/berita-detikhealth/d-0034
35,Tips Menjaga Kesehatan Mata di Era Kerja Serba Digital,https://health.detik.com/berita-detikhealth/d-0035
36,BPOM Awasi Peredaran Obat Tradisional yang Mengandung BKO,https://health.detik.com/berita-detikhealth/d-0036
37,Kenali Gejala Asam Urat dan Cara Menjaga Pola Makan,https://health.detik.com/berita-detikhealth/d-0037
38,Studi: Anak yang Cukup Tidur Miliki Konsentrasi Belajar Lebih Baik,https://health.detik.com/berita-detikhealth/d-0038
39,Pentingnya Vaksin Flu Tahunan bagi Kelompok Rentan,https://health.detik.com/berita-detikhealth/d-0039



Jumlah seluruh kata unik: 210

========== 10 KATA UNIK PERTAMA ==========


,Kata Unik
No,
1,ahli
2,air
3,akut
4,alasan
5,aman
6,anak
7,asam
8,atau
9,awal



========== 10 KATA UNIK TERAKHIR ==========


,Kata Unik
No,
201,untuk
202,urat
203,usus
204,vaksin
205,vaksinasi
206,warga
207,waspada
208,waspadai
209,yang



Data berhasil disimpan ke 'df_berita_hasil_crawling.csv'
